In [ ]:
!pip install -q kagglehub datasets transformers accelerate scikit-learn matplotlib seaborn
!pip install evaluate
import kagglehub
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from datasets import Dataset
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import BaseEstimator, TransformerMixin
import evaluate


In [ ]:
path = kagglehub.dataset_download("jacopoferretti/bbc-articles-dataset")
print(f"Contenido de la ruta de descarga ({path}):")
data_raw = pd.read_csv(os.path.join(path, "bbc_news_text_complexity_summarization.csv"))
data_raw.head()

In [ ]:
data = data_raw.copy()
data.info()


In [ ]:
labels = data["labels"].unique().tolist()
NUM_LABELS = len(labels)
MAX_SEQUENCE_LENGTH = 5
id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

In [ ]:
# Codificacion de variable dependiente de categoria a numerica.
le = LabelEncoder()
data['labels_encoded'] = le.fit_transform(data['labels'])
Y_encoded = data['labels_encoded'].values

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def limpiar_texto(textos: pd.Series) -> pd.Series:
    """
    Realiza la limpieza básica de URLs, IPs y convierte a minúsculas.
    (La normalización de stopwords la manejará BERT internamente).
    """
    return textos.apply(
        lambda t: re.sub(
            r"http\S+|www\S+|https\S+|\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b",
            "",
            str(t).lower()
        )
    )

In [ ]:
class BertSequenceEncoder(BaseEstimator, TransformerMixin):
    """
    Transforma texto limpio en tensores de input_ids y attention_mask
    (tokenización, padding y truncamiento).
    """
    def __init__(self, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len


    def fit(self, X, y=None):
        return self

    def transform(self, X: pd.Series):
        encoded_inputs = self.tokenizer(
        X.tolist(),
        max_length=self.max_len, # <-- Usa el valor 70
        padding='max_length',
        truncation=True,
        return_tensors='np'
        )
        return {
            'input_ids': encoded_inputs['input_ids'],
            'attention_mask': encoded_inputs['attention_mask'],
        }

        # El modelo de clasificación multi-clase usará estas tres entradas (si aplica):
        return {
            'input_ids': encoded_inputs['input_ids'],
            'attention_mask': encoded_inputs['attention_mask'],
            # 'token_type_ids': encoded_inputs.get('token_type_ids', None) # Se incluye si el modelo lo requiere (para tasks como QA)
        }

In [ ]:
bert_pipeline = Pipeline([
    ('limpieza', FunctionTransformer(limpiar_texto)),
    # Se pasa el max_len de 70, haciendo que el pipeline sea viable en Colab
    ('bert_encoder', BertSequenceEncoder(tokenizer=tokenizer, max_len=MAX_SEQUENCE_LENGTH))
])
X_encoded = bert_pipeline.fit_transform(data['text'])

In [ ]:
indices = np.arange(len(data))
# Dataset principal y preubas y el de inferencia

main_idx, inference_idx = train_test_split(
    indices,
    test_size=0.01,
    random_state=42,
    stratify=Y_encoded
)
# Dataset de Entrenamiento/pruebas
train_idx, val_test_idx = train_test_split(
    main_idx,
    test_size=0.2, # 20% del 99%
    random_state=42,
    stratify=Y_encoded[main_idx]
)
# Dataset de Validación y preubas
val_idx, test_idx = train_test_split(
    val_test_idx,
    test_size=0.5, # 50% del 20%
    random_state=42,
    stratify=Y_encoded[val_test_idx]
)

In [ ]:
def slice_and_package_numpy(indices):
    """Devuelve un diccionario de arrays de NumPy con dtype int64 asegurado."""
    return {
        'input_ids': X_encoded['input_ids'][indices].astype(np.int64),
        'attention_mask': X_encoded['attention_mask'][indices].astype(np.int64),
        'labels': Y_encoded[indices].astype(np.int64)
    }
def convert_to_tensors_and_create_dataset(indices):
    """Convierte los arrays NumPy directamente a tensores PyTorch y crea el Dataset."""

    data_dict_np = slice_and_package_numpy(indices)

    # Conversión CRÍTICA: NumPy array -> PyTorch Tensor
    data_dict_tensors = {
        # Usamos torch.from_numpy() para la conversión directa
        'input_ids': torch.from_numpy(data_dict_np['input_ids']),
        'attention_mask': torch.from_numpy(data_dict_np['attention_mask']),
        'labels': torch.from_numpy(data_dict_np['labels']),
    }
    return Dataset.from_dict(data_dict_tensors)
train_tok = convert_to_tensors_and_create_dataset(train_idx)
val_tok = convert_to_tensors_and_create_dataset(val_idx)
test_tok = convert_to_tensors_and_create_dataset(test_idx)
inference_tok = convert_to_tensors_and_create_dataset(inference_idx)

print(f"\n¡Datasets creados con éxito!")
print(f"Dataset Entrenamiento (train_tok): {len(train_tok)} muestras.")
print(f"Dataset Validación (val_tok): {len(val_tok)} muestras.")
print(f"Dataset Prueba (test_tok): {len(test_tok)} muestras.")
print(f"Dataset Inferencia (inference_tok): {len(inference_tok)} muestras.")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
acc = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": acc.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="bbc_distilbert",
    # CORRECCIÓN: Usar 'eval_strategy' en lugar de 'evaluation_strategy'
    eval_strategy="epoch",
    save_strategy="epoch", # Usar 'save_strategy' en lugar de 'save_strategy'
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

In [ ]:
history = trainer.state.log_history

train_log_entries = [x for x in history if "loss" in x and x.get('step') is not None]
train_losses = [x["loss"] for x in train_log_entries]
train_steps = [x["step"] for x in train_log_entries]

eval_log_entries = [x for x in history if "eval_loss" in x and x.get('step') is not None]
eval_losses = [x["eval_loss"] for x in eval_log_entries]
eval_steps = [x["step"] for x in eval_log_entries]
eval_acc = [x["eval_accuracy"] for x in eval_log_entries]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_steps, train_losses, label="Train Loss", color='blue')
plt.plot(eval_steps, eval_losses, label="Validation Loss", color='orange', marker='o', linestyle='--')
plt.title("Loss (Pérdida) vs. Pasos de Entrenamiento")
plt.xlabel("Training Steps (Pasos de Entrenamiento)")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(eval_steps, eval_acc, label="Validation Accuracy", color='green', marker='o')

plt.title("Accuracy (Precisión) vs. Pasos de Entrenamiento")
plt.xlabel("Training Steps (Pasos de Entrenamiento)")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

In [ ]:
preds = trainer.predict(test_tok)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

plt.figure(figsize=(8,6))
disp.plot(xticks_rotation=45, cmap="Blues")
plt.title("Noticias – DistilBERT – Matriz de Confusión")
plt.show()

In [ ]:
# Ejecutar la predicción en el conjunto de inferencia
predictions = trainer.predict(inference_tok)

# Los logits son las salidas crudas del modelo antes de la función softmax.
logits = predictions.predictions

print("Predicción completada.")

In [ ]:
# DEFINICIÓN DEL MAPEO (AJUSTA ESTO A TUS CLASES REALES)
# Si tu dataset es el BBC, tus clases podrían ser estas:
id2label = {
    0: "business",
    1: "entertainment",
    2: "politics",
    3: "sport",
    4: "tech"
}

In [ ]:
def decodificar_texto(ejemplo):
    """
    Función para decodificar los 'input_ids' de un ejemplo de vuelta a texto.
    'ejemplo' es una fila del dataset (un diccionario).
    """
    # Usamos skip_special_tokens=True para omitir [CLS], [SEP], etc.
    texto_reconstruido = tokenizer.decode(ejemplo['input_ids'], skip_special_tokens=True)
    return {"texto_reconstruido": texto_reconstruido}

print("Iniciando decodificación del dataset de inferencia...")
# Aplica la función de decodificación al dataset.
# Esto ASUME que tu dataset (inference_tok) contiene la columna 'input_ids'.
inference_tok_with_text = inference_tok.map(decodificar_texto)
print("Decodificación completada. Columna 'texto_reconstruido' añadida.")

In [ ]:
import numpy as np

def get_final_predictions(logits, id2label):
    """
    Toma los logits del modelo, encuentra la clase con la puntuación más alta
    y mapea el índice a la etiqueta de texto.
    """
    # 1. Encontrar el índice (ID de clase) con la puntuación más alta para cada muestra
    predicted_ids = np.argmax(logits, axis=1)

    # 2. Mapear el ID de clase (0, 1, 2...) a la etiqueta de texto ("business", "sport"...)
    predicted_labels = [id2label[id] for id in predicted_ids]

    return predicted_ids, predicted_labels

# Obtener las predicciones finales
predicted_ids, predicted_labels = get_final_predictions(logits, id2label)

# Opcional: Obtener las etiquetas reales del dataset de inferencia (para comparacion)
true_ids = np.array(list(inference_tok["labels"]))
true_labels = [id2label[id] for id in true_ids]

print("\n--- Resultados de Inferencia (Primeras 5 Muestras) ---")
for i in range(22):
    texto_muestra = inference_tok_with_text[i]["texto_reconstruido"]
    print(f"Muestra {i+1}:")
    print(f"  Texto: {texto_muestra[:100]}... (Mostrar solo los primeros 100 caracteres)") # Muestra el texto
    print(f"  Etiqueta REAL (ID): {true_ids[i]} / {true_labels[i]}")
    print(f"  Etiqueta PREDICHA (ID): {predicted_ids[i]} / {predicted_labels[i]}")

# Calcular la precisión global en el conjunto de inferencia
accuracy = np.mean(predicted_ids == true_ids)
print(f"\nPrecisión total en el conjunto de inferencia: {accuracy*100:.2f}%")

In [ ]:
def decodificar_texto(ejemplo):
    """
    Decodifica la lista de 'input_ids' de vuelta a texto legible.
    """
    # Usamos skip_special_tokens=True para omitir [CLS], [SEP], etc.
    texto_reconstruido = tokenizer.decode(ejemplo['input_ids'], skip_special_tokens=True)
    return {"texto_reconstruido": texto_reconstruido}

In [ ]:
# Aplica la función a todo tu dataset
# Esto añadirá una nueva columna llamada 'texto_reconstruido'
train_tok_con_texto = train_tok.map(decodificar_texto)

# Muestra el resultado
print(train_tok_con_texto[0]["texto_reconstruido"])